# Step 0: Cleaning & Filtering the Raw Job Postings Dataset

This notebook takes the raw LinkedIn Job Postings dataset (`postings.csv`, ~123,850 rows)
and produces a cleaned dataset filtered down to the three roles this project focuses on:
**Sales, Project Manager, and Marketing**.

Steps performed:
1. Load the raw data and select the columns we need
2. Remove exact-duplicate / missing descriptions
3. Filter postings to the target roles based on job title
4. Map LinkedIn's experience-level field onto our Junior / Senior variable
5. Light text cleaning of the job description field
6. Save the cleaned, filtered dataset for use in the next notebook (dictionary expansion)

> **Note:** Update `RAW_PATH` below to point to wherever you've saved `postings.csv`
> from the `archive.zip` dataset.


In [1]:
import pandas as pd
import re

RAW_PATH = "postings.csv"          # path to the raw LinkedIn postings dataset
OUTPUT_PATH = "cleaned_filtered_jobs.csv"

pd.set_option("display.max_colwidth", 80)


## 1. Load the raw data

We only need a handful of columns for this stage: the job id, title, description,
experience level, location and company name.


In [2]:
COLS = ['job_id', 'title', 'description', 'formatted_experience_level',
        'location', 'company_name']

df = pd.read_csv(RAW_PATH, usecols=COLS)
print(f"Loaded {len(df):,} rows")
df.head(3)


Loaded 123,849 rows


,job_id,company_name,title,description,location,formatted_experience_level
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in New Jersey is seeking an admini...,"Princeton, NJ",NaN
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committed to serving clients with bes...","Fort Collins, CO",NaN
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting applications for an Assistant Restaurant ...,"Cincinnati, OH",NaN


## 2. Remove exact duplicates and missing descriptions

Some postings share identical description text (e.g. the same ad re-posted by a company).
We drop exact duplicates on the `description` column, and drop any rows missing a title
or description since we can't classify or analyse those.


In [3]:
before = len(df)

df = df.dropna(subset=['title', 'description'])
df = df.drop_duplicates(subset=['description'], keep='first')

after = len(df)
print(f"Dropped {before - after:,} rows ({before:,} -> {after:,}) "
      f"due to missing or duplicate descriptions")


Dropped 16,022 rows (123,849 -> 107,827) due to missing or duplicate descriptions


## 3. Filter to target roles (Sales / Project Manager / Marketing)

We tag each posting based on whether its **title** contains one of our target role
keywords (case-insensitive substring match). A posting can match more than one role
(e.g. "Marketing and Sales Coordinator") — these ambiguous postings are kept aside
and excluded from the single-role dataset, since they can't be cleanly assigned to
one framing condition.


In [4]:
ROLE_PATTERNS = {
    'sales': 'sales',
    'project_manager': r'project\s+manager',
    'business_analyst': r'business\s+analyst',
    'marketing': 'marketing',
}

for role, pattern in ROLE_PATTERNS.items():
    df[f'is_{role}'] = df['title'].str.contains(pattern, case=False, regex=True, na=False)

match_cols = [f'is_{r}' for r in ROLE_PATTERNS]
df['n_role_matches'] = df[match_cols].sum(axis=1)

print("Postings matching each role keyword (title contains):")
for role in ROLE_PATTERNS:
    print(f"  {role:>16}: {df[f'is_{role}'].sum():,}")

print(f"\nPostings matching more than one role keyword: {(df['n_role_matches'] > 1).sum():,}")
print(f"Postings matching no role keyword:             {(df['n_role_matches'] == 0).sum():,}")


Postings matching each role keyword (title contains):
             sales: 6,185
   project_manager: 1,870
  business_analyst: 534
         marketing: 1,699

Postings matching more than one role keyword: 211
Postings matching no role keyword:             97,750


In [5]:
def assign_role(row):
    matches = [r for r in ROLE_PATTERNS if row[f'is_{r}']]
    if len(matches) == 1:
        return matches[0]
    elif len(matches) == 0:
        return None
    else:
        return 'multiple'

df['role'] = df.apply(assign_role, axis=1)

filtered = df[df['role'].isin(ROLE_PATTERNS.keys())].copy()
filtered = filtered.drop(columns=match_cols + ['n_role_matches'])

print(f"Kept {len(filtered):,} single-role postings out of {len(df):,}")
filtered['role'].value_counts()


Kept 9,866 single-role postings out of 107,827


role
sales               6004
project_manager     1833
marketing           1519
business_analyst     510
Name: count, dtype: int64

## 4. Map seniority (Junior / Senior)

LinkedIn's `formatted_experience_level` field has six categories plus missing values.
We map these onto the binary Junior/Senior variable from your study design. This
mapping is a judgement call — feel free to adjust which levels count as "junior" vs
"senior" for your study. Postings with no listed experience level are kept but marked
as `NaN`, so you can decide later whether to drop them or try to infer seniority from
the description text.


In [6]:
SENIORITY_MAP = {
    'Internship': 'junior',
    'Entry level': 'junior',
    'Associate': 'junior',
    'Mid-Senior level': 'senior',
    'Director': 'senior',
    'Executive': 'senior',
}

filtered['seniority'] = filtered['formatted_experience_level'].map(SENIORITY_MAP)

print(filtered['seniority'].value_counts(dropna=False))
print()
print("Role x Seniority breakdown:")
pd.crosstab(filtered['role'], filtered['seniority'], dropna=False)


seniority
senior    3564
junior    3517
NaN       2785
Name: count, dtype: int64

Role x Seniority breakdown:


seniority,junior,senior,NaN
role,,,
business_analyst,72,282,156
marketing,436,593,490
project_manager,368,1059,406
sales,2641,1630,1733


## 5. Clean the job description text

Two light cleaning steps:

- **Collapse whitespace**: many descriptions contain repeated newlines/spaces from the
  original HTML formatting.
- **Split glued words**: some descriptions have section headers glued directly onto
  the following sentence with no space (e.g. `"Job descriptionA leading firm..."`),
  which happens when HTML tags were stripped without inserting a space. We insert a
  space wherever a lowercase letter is immediately followed by an uppercase letter.

This second step is a heuristic — it occasionally over-splits things like acronyms or
brand names (e.g. "iPhone" -> "i Phone"), but for word-frequency and embedding-based
dictionary expansion (the next notebook), the benefit of un-gluing run-on words
outweighs this minor noise. If you need exact text later (e.g. for the cover-letter
generation), keep a copy of the original `description` column too.


In [7]:
GLUED_WORD_PATTERN = re.compile(r'([a-z])([A-Z])')

def clean_description(text):
    text = GLUED_WORD_PATTERN.sub(r'\1 \2', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

filtered['description_clean'] = filtered['description'].apply(clean_description)

# Show a before/after example for one of the postings affected by glued words
example = filtered[filtered['description'].str.startswith('Job description', na=False)].iloc[0]
print("BEFORE:", example['description'][:150])
print()
print("AFTER: ", example['description_clean'][:150])


BEFORE: Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You

AFTER:  Job description A leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. Yo


## 6. Save the cleaned, filtered dataset

We keep the original `description` alongside the cleaned version, plus the role and
seniority labels, ready for the next notebook (dictionary expansion).


In [8]:
final_cols = ['job_id', 'title', 'role', 'seniority', 'location', 'company_name',
              'description', 'description_clean']
final_df = filtered[final_cols].reset_index(drop=True)

final_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(final_df):,} cleaned, filtered postings to '{OUTPUT_PATH}'")
print()
print("Final breakdown by role and seniority:")
pd.crosstab(final_df['role'], final_df['seniority'], dropna=False, margins=True)


Saved 9,866 cleaned, filtered postings to 'cleaned_filtered_jobs.csv'

Final breakdown by role and seniority:


seniority,junior,senior,NaN,All
role,,,,
business_analyst,72,282,156,510
marketing,436,593,490,1519
project_manager,368,1059,406,1833
sales,2641,1630,1733,6004
All,3517,3564,2785,9866
